# Free MuseTalk 1.5 lip-sync test (Kaggle GPU)

This private preview runs the open-source MuseTalk 1.5 model on Kaggle's free NVIDIA T4 GPU pool. GitHub Actions creates a private Kaggle Dataset with one existing Kling presenter clip and one matching narration track, then submits this notebook as a private GPU kernel.

The preview accepts a `bbox_shift` mouth-openness setting (default +4, range -9 to +9); positive values generally increase mouth openness. The current sample uses a moderate shift to avoid an over-open or distorted result.\n\nThe repository workflow uses the GitHub Actions secret `KAGGLE_API_TOKEN` and discovers the Kaggle account from the token. The workflow is manually triggerable and can also run the first preview after its narrowly scoped merge trigger. It never publishes to YouTube.

**Budget: USD 0.** The submission explicitly requests Kaggle's `NvidiaTeslaT4` free GPU pool. If Kaggle cannot attach a GPU, the first cell stops immediately; no CPU fallback and no paid service are used. Keep the first preview short (about 4–8 seconds). Do not put credentials in the dataset.


In [ ]:
import glob, os, shutil, subprocess, torch

print("Kaggle base-image PyTorch:", torch.__version__)
print("Base-image CUDA visibility:", torch.cuda.is_available())

# Some Kaggle base images omit the nvidia-smi binary. Check device nodes instead,
# then validate real CUDA access after installing the dedicated MuseTalk CUDA wheel.
gpu_nodes = sorted(glob.glob("/dev/nvidia*"))
print("NVIDIA device nodes:", gpu_nodes)
if not any(os.path.basename(p).startswith("nvidia") and p.endswith(("0", "ctl")) for p in gpu_nodes):
    raise RuntimeError(
        "Kaggle accepted the GPU notebook metadata but did not mount an NVIDIA device. "
        "Free GPU quota/capacity may be unavailable. Stop here; do not use CPU or paid fallback."
    )

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    probe = subprocess.run([nvidia_smi, "-L"], capture_output=True, text=True)
    print("nvidia-smi:", probe.stdout.strip() or probe.stderr.strip())
else:
    print("nvidia-smi executable is absent; the CUDA-enabled MuseTalk venv will verify the GPU in the install cell.")


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq ffmpeg
git clone https://github.com/TMElyralab/MuseTalk.git /kaggle/working/MuseTalk
cd /kaggle/working/MuseTalk

# Upstream MuseTalk pins NumPy/TensorFlow to versions that do not support
# Kaggle's current Python 3.13 image. Use the project's recommended Python 3.10.
python -m pip install -q --upgrade uv
uv python install 3.10
# Seed the environment with pip/setuptools/wheel. uv otherwise creates a
# pip-free environment, so "python -m pip" would fail on the next line.
uv venv --seed /kaggle/working/musetalk-venv --python 3.10
MUSEPY=/kaggle/working/musetalk-venv/bin/python
MUSEBIN=/kaggle/working/musetalk-venv/bin
"$MUSEPY" -m pip --version
"$MUSEPY" -m pip install -q --upgrade pip setuptools wheel

# MuseTalk's documented CUDA 11.8 build supports Kaggle's NVIDIA T4 pool.
"$MUSEPY" -m pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 \
  --index-url https://download.pytorch.org/whl/cu118
"$MUSEPY" -m pip install -q -r requirements.txt
# The upstream MuseTalk checkout contains source files under musetalk/whisper,
# but that folder has no setup.py or pyproject.toml. Inference uses the
# Transformers WhisperModel and does not require pip-installing that folder.

# Install the OpenMMLab runtime components required by MuseTalk's DWPose
# preprocessing. These match the upstream MuseTalk installation guidance.
"$MUSEPY" -m pip install -q --no-cache-dir -U openmim
"$MUSEBIN/mim" install -q mmengine
"$MUSEBIN/mim" install -q "mmcv==2.0.1"
"$MUSEBIN/mim" install -q "mmdet==3.1.0"
# MMPose declares chumpy (legacy SMPL fitting) as a setup dependency; MuseTalk DWPose inference does not use it.
# Install MMPose without optional/legacy deps to avoid chumpy failing to build on modern pip.
"$MUSEPY" -m pip install -q --no-deps "mmpose==1.1.0"
"$MUSEPY" -m pip install -q json_tricks munkres xtcocotools yapf
"$MUSEPY" -m pip install -q huggingface_hub==0.30.2 gdown

# MuseTalk's preprocessing imports its bundled face_detection package by
# its top-level module name; expose that source folder as well as the repo root.
export PYTHONPATH="$PWD/musetalk/utils:$PWD${PYTHONPATH:+:$PYTHONPATH}"
# Kaggle sets an IPython-only Matplotlib backend. MuseTalk runs in a separate
# Python 3.10 environment, so force a headless backend before importing MMPose.
export MPLBACKEND=Agg
export MPLCONFIGDIR=/kaggle/working/matplotlib-config
mkdir -p "$MPLCONFIGDIR"

# Fail here with a clear message instead of later during video inference.
"$MUSEPY" - <<'PY'
import torch, matplotlib
if matplotlib.get_backend().lower() != "agg":
    raise RuntimeError(f"Expected Matplotlib Agg backend, got {matplotlib.get_backend()}")
print("Matplotlib backend:", matplotlib.get_backend())
import mmcv, mmdet, mmpose
from transformers import WhisperModel
from musetalk.utils.audio_processor import AudioProcessor
from face_detection import FaceAlignment, LandmarksType
from mmpose.apis import inference_topdown, init_model

print("MuseTalk PyTorch:", torch.__version__)
print("MuseTalk CUDA:", torch.cuda.is_available())
print("MuseTalk GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("mmcv:", mmcv.__version__)
print("mmdet:", mmdet.__version__)
print("mmpose:", mmpose.__version__)
print("WhisperModel, AudioProcessor and face_detection imports: OK")
if not torch.cuda.is_available():
    raise RuntimeError("MuseTalk Python 3.10 environment cannot see the free Kaggle GPU.")
PY

# Prove the dedicated Python environment can see the requested GPU.
"$MUSEPY" - <<'PY'
import torch
print("MuseTalk environment PyTorch:", torch.__version__)
print("MuseTalk environment CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("MuseTalk Python 3.10 environment cannot see the free Kaggle GPU.")
print("MuseTalk GPU:", torch.cuda.get_device_name(0))
PY

# The upstream downloader otherwise upgrades huggingface_hub without a pin.
python - <<'PY'
from pathlib import Path
p = Path("download_weights.sh")
s = p.read_text()
s = s.replace(
    'pip install -U "huggingface_hub[cli]"',
    'pip install "huggingface_hub[cli]==0.30.2"'
)
p.write_text(s)
PY

echo "MuseTalk source and Python 3.10 CUDA dependencies installed."


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/MuseTalk

# Replace the upstream non-fail-fast model download script. Every required
# V1.5 asset is downloaded directly, retried, size-checked and hash-checked.

download_verified() {
  local dest="$1" min_bytes="$2" expected_sha="$3"
  shift 3
  local tmp="$dest.part" url bytes actual
  mkdir -p "$(dirname "$dest")"

  if [[ -s "$dest" ]]; then
    bytes="$(stat -c%s "$dest")"
    if [[ "$bytes" -ge "$min_bytes" ]]; then
      if [[ "$expected_sha" == "-" ]] || [[ "$(sha256sum "$dest" | awk '{print $1}')" == "$expected_sha" ]]; then
        echo "Verified cached asset: $dest ($bytes bytes)"
        return 0
      fi
    fi
    rm -f "$dest"
  fi

  for url in "$@"; do
    echo "Downloading $dest from $url"
    rm -f "$tmp"
    if curl --fail --location --show-error --progress-bar \
      --retry 4 --retry-delay 3 --retry-all-errors \
      --connect-timeout 30 --max-time 7200 \
      --output "$tmp" "$url"; then
      bytes="$(stat -c%s "$tmp")"
      if [[ "$bytes" -lt "$min_bytes" ]]; then
        echo "Rejected $url: $bytes bytes; expected at least $min_bytes." >&2
        rm -f "$tmp"
        continue
      fi
      if [[ "$expected_sha" != "-" ]]; then
        actual="$(sha256sum "$tmp" | awk '{print $1}')"
        if [[ "$actual" != "$expected_sha" ]]; then
          echo "Rejected $url: SHA256 mismatch ($actual)." >&2
          rm -f "$tmp"
          continue
        fi
      fi
      mv "$tmp" "$dest"
      echo "Verified $dest ($bytes bytes)"
      return 0
    fi
    echo "Download source failed: $url" >&2
  done

  echo "::error::Could not download and verify required model asset: $dest" >&2
  return 1
}

# MuseTalk V1.5 UNet (primary and duplicate mirrors share this SHA256).
download_verified models/musetalkV15/unet.pth 3400000000 \
  7ebf6c98c181e20838e4c0054e96e944ac60d5d692cc01db42839fe11b787007 \
  "https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/unet.pth?download=true" \
  "https://huggingface.co/skye1128/MuseTalk/resolve/main/musetalkV15/unet.pth?download=true" \
  "https://hf-mirror.com/TMElyralab/MuseTalk/resolve/main/musetalkV15/unet.pth"

download_verified models/musetalkV15/musetalk.json 700 - \
  "https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json?download=true" \
  "https://huggingface.co/skye1128/MuseTalk/resolve/main/musetalkV15/musetalk.json?download=true" \
  "https://hf-mirror.com/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json"

# Stable Diffusion VAE (model loader expects models/sd-vae).
download_verified models/sd-vae/config.json 500 - \
  "https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/config.json?download=true" \
  "https://hf-mirror.com/stabilityai/sd-vae-ft-mse/resolve/main/config.json"
download_verified models/sd-vae/diffusion_pytorch_model.bin 300000000 \
  1b4889b6b1d4ce7ae320a02dedaeff1780ad77d415ea0d744b476155c6377ddc \
  "https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin?download=true" \
  "https://huggingface.co/diffusers/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin?download=true" \
  "https://hf-mirror.com/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin"

# Whisper tiny encoder.
download_verified models/whisper/config.json 500 - \
  "https://huggingface.co/openai/whisper-tiny/resolve/main/config.json?download=true" \
  "https://hf-mirror.com/openai/whisper-tiny/resolve/main/config.json"
download_verified models/whisper/preprocessor_config.json 100 - \
  "https://huggingface.co/openai/whisper-tiny/resolve/main/preprocessor_config.json?download=true" \
  "https://hf-mirror.com/openai/whisper-tiny/resolve/main/preprocessor_config.json"
download_verified models/whisper/pytorch_model.bin 140000000 \
  9607f98a2b22d9e229ae43c52ecea79dcede9e0c5cfae67e8da6eda86d8aac1d \
  "https://huggingface.co/openai/whisper-tiny/resolve/main/pytorch_model.bin?download=true" \
  "https://hf-mirror.com/openai/whisper-tiny/resolve/main/pytorch_model.bin"

# DWPose; fallback mirrors resolve to the same verified 407 MB checkpoint.
download_verified models/dwpose/dw-ll_ucoco_384.pth 400000000 \
  0d9408b13cd863c4e95a149dd31232f88f2a12aa6cf8964ed74d7d97748c7a07 \
  "https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth?download=true" \
  "https://huggingface.co/camenduru/DWPose/resolve/main/dw-ll_ucoco_384.pth?download=true" \
  "https://huggingface.co/camenduru/MuseTalk/resolve/5cfe77e50e1dc10ac599e5340d610d69a5086af2/dwpose/dw-ll_ucoco_384.pth?download=true" \
  "https://hf-mirror.com/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth"

# Face parser and ResNet18 backbone.
download_verified models/face-parse-bisent/79999_iter.pth 50000000 \
  468e13ca13a9b43cc0881a9f99083a430e9c0a38abd935431d1c28ee94b26567 \
  "https://huggingface.co/ManyOtherFunctions/face-parse-bisent/resolve/9b440738b337fd51492d718685e452af3532acd0/79999_iter.pth?download=true" \
  "https://hf-mirror.com/ManyOtherFunctions/face-parse-bisent/resolve/main/79999_iter.pth"
download_verified models/face-parse-bisent/resnet18-5c106cde.pth 40000000 - \
  "https://download.pytorch.org/models/resnet18-5c106cde.pth"

# Preload SFD detector. Upstream otherwise downloads it silently during inference.
download_verified musetalk/utils/face_detection/detection/sfd/s3fd.pth 80000000 \
  619a31681264d3f7f7fc7a16a42cbbe8b23f31a256f75a366e5a1bcd59b33543 \
  "https://huggingface.co/ByteDance/LatentSync/resolve/main/auxiliary/s3fd-619a316812.pth?download=true" \
  "https://huggingface.co/wsj1995/sadTalker/resolve/main/s3fd-619a316812.pth?download=true" \
  "https://hf-mirror.com/ByteDance/LatentSync/resolve/main/auxiliary/s3fd-619a316812.pth" \
  "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth"

# Do not let inference run with incomplete or HTML/error-page model files.
python - <<'PY'
import json
from pathlib import Path

required = {
    "models/musetalkV15/unet.pth": 3_400_000_000,
    "models/musetalkV15/musetalk.json": 700,
    "models/sd-vae/config.json": 500,
    "models/sd-vae/diffusion_pytorch_model.bin": 300_000_000,
    "models/whisper/config.json": 500,
    "models/whisper/preprocessor_config.json": 100,
    "models/whisper/pytorch_model.bin": 140_000_000,
    "models/dwpose/dw-ll_ucoco_384.pth": 400_000_000,
    "models/face-parse-bisent/79999_iter.pth": 50_000_000,
    "models/face-parse-bisent/resnet18-5c106cde.pth": 40_000_000,
    "musetalk/utils/face_detection/detection/sfd/s3fd.pth": 80_000_000,
}
for name, minimum in required.items():
    p = Path(name)
    if not p.is_file() or p.stat().st_size < minimum:
        raise SystemExit(f"Required model asset missing/truncated: {name}")
    print(f"PRECHECK OK: {name} ({p.stat().st_size:,} bytes)")

for name in (
    "models/musetalkV15/musetalk.json",
    "models/sd-vae/config.json",
    "models/whisper/config.json",
    "models/whisper/preprocessor_config.json",
):
    with open(name, encoding="utf-8") as f:
        json.load(f)
print("All required V1.5 model assets passed size, checksum and JSON checks.")
PY


In [ ]:
# Find one video, one matching audio file, and optional mouth-openness settings.
from pathlib import Path
import os, glob, shutil, json

video_candidates = [p for p in glob.glob('/kaggle/input/**/*', recursive=True) if p.lower().endswith(('.mp4','.mov','.mkv'))]
audio_candidates = [p for p in glob.glob('/kaggle/input/**/*', recursive=True) if p.lower().endswith(('.wav','.mp3','.m4a'))]
settings_candidates = glob.glob('/kaggle/input/**/lip_sync_settings.json', recursive=True)
print('Video files:', video_candidates)
print('Audio files:', audio_candidates)
print('Lip-sync settings:', settings_candidates)

if len(video_candidates) != 1 or len(audio_candidates) != 1:
    raise RuntimeError('Attach a private Kaggle Dataset containing exactly one test video and one matching narration audio file.')
if len(settings_candidates) > 1:
    raise RuntimeError('Attach at most one lip_sync_settings.json file to avoid ambiguous settings.')

work = Path('/kaggle/working/MuseTalk')
video_path = Path(video_candidates[0])
audio_path = Path(audio_candidates[0])
data_dir = work / 'data' / 'free_test'
data_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(video_path, data_dir / 'presenter.mp4')
shutil.copy2(audio_path, data_dir / ('narration' + audio_path.suffix))

# A moderate positive shift opens the generated mouth area without applying
# any post-render stretching or changing the face crop's aspect ratio.
bbox_shift = 4
if settings_candidates:
    settings = json.loads(Path(settings_candidates[0]).read_text())
    bbox_shift = int(settings.get('bbox_shift', bbox_shift))
if not -9 <= bbox_shift <= 9:
    raise RuntimeError(f'bbox_shift must be in MuseTalk\'s documented -9 to +9 range; got {bbox_shift}.')
print(f'Using MuseTalk bbox_shift={bbox_shift} (positive values generally increase mouth openness).')
print('Input staged:', list(data_dir.iterdir()))

In [ ]:
# Configure one inference task. MuseTalk recommends 25 fps input for best results.
import yaml
config = {
  'task_0': {
    'video_path': str(data_dir / 'presenter.mp4'),
    'audio_path': str(data_dir / ('narration' + audio_path.suffix)),
    'bbox_shift': bbox_shift
  }
}
config_path = work / 'configs' / 'inference' / 'kaggle_free_test.yaml'
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w') as f:
    yaml.safe_dump(config, f, sort_keys=False)
print(f'Inference config uses bbox_shift={bbox_shift}:')
print(config_path.read_text())

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/MuseTalk
export PYTHONPATH="$PWD/musetalk/utils:$PWD${PYTHONPATH:+:$PYTHONPATH}"
export MPLBACKEND=Agg
export MPLCONFIGDIR=/kaggle/working/matplotlib-config
mkdir -p "$MPLCONFIGDIR"
/kaggle/working/musetalk-venv/bin/python -m scripts.inference \
  --inference_config configs/inference/kaggle_free_test.yaml \
  --result_dir /kaggle/working/musetalk_preview \
  --unet_model_path models/musetalkV15/unet.pth \
  --unet_config models/musetalkV15/musetalk.json \
  --version v15 \
  --fps 25 \
  --batch_size 4 \
  --use_float16 \
  --output_vid_name model_render.mp4


In [ ]:
# Export one known MuseTalk result to a stable, root-level filename.
# This lets the GitHub workflow retrieve the preview without downloading the
# cloned repository, model checkpoints, Python environment or example videos.
from pathlib import Path
from IPython.display import Video, display
import json
import shutil
import subprocess

rendered = Path('/kaggle/working/musetalk_preview/v15/model_render.mp4')
if not rendered.is_file():
    candidates = sorted(str(p) for p in Path('/kaggle/working/musetalk_preview').rglob('*.mp4'))
    raise RuntimeError(
        'MuseTalk did not create the expected v15/model_render.mp4. '
        f'Other MP4 candidates: {candidates[:10]}. No unsynchronized fallback will be used.'
    )

probe = json.loads(subprocess.check_output([
    'ffprobe', '-v', 'error', '-show_streams', '-show_format', '-of', 'json', str(rendered)
], text=True))
stream_types = {stream.get('codec_type') for stream in probe.get('streams', [])}
duration = float(probe.get('format', {}).get('duration', 0))
size = rendered.stat().st_size
if size < 100_000 or not 2 <= duration <= 30:
    raise RuntimeError(f'Invalid MuseTalk output size/duration: {size} bytes, {duration:.2f}s')
if not {'video', 'audio'}.issubset(stream_types):
    raise RuntimeError(f'MuseTalk output is missing audio or video: {sorted(stream_types)}')

exported = Path('/kaggle/working/musetalk_preview.mp4')
shutil.copy2(rendered, exported)
print(json.dumps({
    'export': str(exported),
    'source': str(rendered),
    'bytes': exported.stat().st_size,
    'duration_seconds': duration,
    'streams': sorted(stream_types)
}, indent=2))
display(Video(str(exported), embed=True))
